# Day 13 · Exercise 5: Answer with Grounding

**What you'll build:** the final piece of the RAG pipeline — a function that retrieves context, builds the augmented prompt, adds a grounding system instruction, and calls the LLM.

**Why it matters:** grounding is what makes RAG trustworthy — without it, the model blends retrieved evidence with its own training weights, making answers hard to trace.

## Your Implementation

In [ ]:
import chromadb
import ollama

# Dependencies from earlier exercises (provided):
def chunk_document(text: str, chunk_size: int = 400, overlap: int = 50) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be less than chunk_size")
    chunks, step, start = [], chunk_size - overlap, 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += step
    return chunks

def build_rag_prompt(question: str, chunks: list[str]) -> str:
    context = "\n\n---\n\n".join(chunks)
    return (
        f"Use the following context to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

def retrieve_context(question: str, collection, top_k: int = 3) -> list[str]:
    q_emb = ollama.embeddings(model="nomic-embed-text", prompt=question)
    results = collection.query(query_embeddings=[q_emb["embedding"]], n_results=top_k)
    return results["documents"][0]

GROUNDING_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions strictly from the "
    "provided context. If the answer is not in the context, respond with: "
    "'I don't know based on the provided documents.' "
    "Do not use any knowledge outside the context."
)


def rag_answer(question: str, collection, model: str = "llama3.2") -> str:
    """Full RAG pipeline: retrieve relevant chunks, build grounded prompt, call LLM.

    Args:
        question: The user's question.
        collection: A ChromaDB Collection with indexed document chunks.
        model: Ollama model name to use for generation.

    Returns:
        The model's answer as a string, grounded in the retrieved context.

    Example:
        >>> # (assumes collection is already indexed)
        >>> answer = rag_answer("What is Python?", collection)
        >>> isinstance(answer, str) and len(answer) > 0
        True
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work
Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score = 0
    total = 4

    # Setup: build tiny indexed collection
    try:
        import chromadb as _chromadb
        _client = _chromadb.Client()
        _coll = _client.get_or_create_collection("ex05_test")
        _doc_text = (
            "Python is a high-level, general-purpose programming language. "
            "It emphasises code readability. Guido van Rossum created Python. "
        ) * 3
        for _i, _chunk in enumerate(chunk_document(_doc_text, chunk_size=150, overlap=20)):
            _emb = ollama.embeddings(model="nomic-embed-text", prompt=_chunk)
            _coll.add(
                ids=[f"chunk_{_i:04d}"],
                embeddings=[_emb["embedding"]],
                documents=[_chunk],
                metadatas=[{"source": "python.txt", "chunk_index": _i}],
            )
    except Exception as e:
        print(f"{_FAIL} Setup: could not build test collection: {e}")
        return

    # Check 1: returns a string
    try:
        result = rag_answer("Who created Python?", _coll)
        if not isinstance(result, str):
            print(f"{_FAIL} Check 1: rag_answer should return a str, got {type(result).__name__}")
            return
        print(f"{_PASS} Check 1: rag_answer returns a string")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1: raised {type(e).__name__}: {e}")
        return

    # Check 2: answer is non-empty
    if len(result.strip()) > 0:
        print(f"{_PASS} Check 2: answer is non-empty")
        score += 1
    else:
        print(f"{_FAIL} Check 2: answer is empty")
        return

    # Check 3: answer contains "Guido" or "I don't know" (either is correct grounded behaviour)
    if "Guido" in result or "don't know" in result.lower() or "do not know" in result.lower():
        print(f"{_PASS} Check 3: answer is grounded (mentions creator or admits uncertainty)")
        score += 1
    else:
        print(f"{_FAIL} Check 3: answer doesn't appear grounded — got: '{result[:80]}'")

    # Check 4: out-of-context question returns "I don't know" style response
    unk = rag_answer("What is the population of Mars?", _coll)
    if "don't know" in unk.lower() or "do not know" in unk.lower() or "not in" in unk.lower() or "cannot" in unk.lower():
        print(f"{_PASS} Check 4: out-of-context question triggers 'I don't know' response")
        score += 1
    else:
        print(f"{_FAIL} Check 4: expected 'I don't know' for out-of-context question, got: '{unk[:80]}'")

    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed.")

_run_checks()

## Bonus Challenge
Extend `rag_answer` to return a `dict` with keys `answer` (str) and `chunks_used` (list[str]) so the caller can see which evidence supported the response. This is the foundation of source citations — the feature you'll build fully on Day 14.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def rag_answer(question: str, collection, model: str = "llama3.2") -> str:
    chunks = retrieve_context(question, collection, top_k=3)
    user_msg = build_rag_prompt(question, chunks)
    messages = [
        {"role": "system", "content": GROUNDING_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    response = ollama.chat(model=model, messages=messages)
    return response["message"]["content"]
```

**Why this works:** the grounding system prompt is the constraint — it tells the model to use only the context and to say "I don't know" rather than fabricate. The user message (built by `build_rag_prompt`) carries the actual retrieved evidence. This two-message structure keeps instructions and data cleanly separated.
</details>